In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [6]:
# 1. Load top 30 stock symbols from stocklisttop30.csv
df_stocklist = pd.read_csv('stocklisttop30.csv')
symbols = df_stocklist['Symbol'].tolist()

results = []

for symbol in symbols:
    try:
        print(f"Processing {symbol}...")
        df = yf.download(symbol, start="2023-01-01", end="2026-01-01", auto_adjust=True, progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df = df.droplevel('Ticker', axis=1)
        if df.empty or len(df) < 100:
            continue
            
        df['Close'] = df['Close'].bfill().ffill()
        scaler = MinMaxScaler()
        scaled_close = scaler.fit_transform(df[['Close']])
        
        def create_dataset(dataset, look_back=30):
            dataX, dataY = [], []
            for i in range(len(dataset) - look_back):
                dataX.append(dataset[i:(i + look_back), 0])
                dataY.append(dataset[i + look_back, 0])
            return np.array(dataX), np.array(dataY)
            
        X, Y = create_dataset(scaled_close, look_back=30)
        if len(X) < 50:
            continue
            
        # Train / Test Split (80:20)
        train_size = int(len(X) * 0.8)
        trainX, testX = X[:train_size], X[train_size:]
        trainY, testY = Y[:train_size], Y[train_size:]
        
        trainX = np.reshape(trainX, (trainX.shape[0], trainX.shape[1], 1))
        testX = np.reshape(testX, (testX.shape[0], testX.shape[1], 1))
        
        # LSTM Model Architecture
        model = Sequential([  
            LSTM(units=32, return_sequences=True, input_shape=(trainX.shape[1], 1)),  
            Dropout(0.2),  
            LSTM(units=32, return_sequences=False),  
            Dropout(0.2),  
            Dense(units=16),  
            Dense(units=1)  
        ])  
        model.compile(optimizer='adam', loss='mean_squared_error')
        model.fit(trainX, trainY, epochs=5, batch_size=32, verbose=0)
        
        # Predictions & Inverse Scaling
        testPredict = model.predict(testX, verbose=0)
        testPredict = scaler.inverse_transform(testPredict)
        testY_inv = scaler.inverse_transform(testY.reshape(-1, 1))
        
        # Evaluation Metrics (including R-squared)
        mae = mean_absolute_error(testY_inv, testPredict)
        rmse = np.sqrt(mean_squared_error(testY_inv, testPredict))
        mape = np.mean(np.abs((testY_inv - testPredict) / testY_inv)) * 100
        r2 = r2_score(testY_inv, testPredict)
        
        results.append({
            'Symbol': symbol,
            'MAE': mae,
            'RMSE': rmse,
            'MAPE (%)': mape,
            'R-squared': r2
        })
        print(f"Finished {symbol}: MAE={mae:.4f}, RMSE={rmse:.4f}, MAPE={mape:.2f}%, R2={r2:.4f}")
    except Exception as e:
        print(f"Error for {symbol}: {e}")

print("***completed")

Processing 0700.HK...


C:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished 0700.HK: MAE=15.3976, RMSE=18.0726, MAPE=2.62%, R2=0.8769
Processing 0005.HK...


C:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished 0005.HK: MAE=2.4534, RMSE=3.1855, MAPE=2.46%, R2=0.8306
Processing 1398.HK...


...

Finished 0728.HK: MAE=0.1452, RMSE=0.1706, MAPE=2.72%, R2=0.3231
Processing 0992.HK...


C:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished 0992.HK: MAE=0.5541, RMSE=0.6601, MAPE=5.24%, R2=0.5251
Processing 0267.HK...


C:\Users\USER\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Finished 0267.HK: MAE=0.3376, RMSE=0.4538, MAPE=3.08%, R2=0.4315


In [7]:
# 2. Save comprehensive results to CSV
df_results = pd.DataFrame(results)
df_results.to_csv('lstm_metrics_top30.csv', index=False)
print("Saved final metrics to lstm_metrics_top30.csv")
print(df_results)

Saved final metrics to lstm_metrics_top30.csv
     Symbol        MAE       RMSE  MAPE (%)  R-squared
0   0700.HK  15.397582  18.072614  2.623631   0.876918
1   0005.HK   2.453393   3.185499  2.461369   0.830649
2   1398.HK   0.140130   0.169392  2.488983   0.436666
3   0939.HK   0.222648   0.258192  2.996459   0.204983
4   3750.HK  25.217728  28.005186  5.128664  -1.854508
5   9988.HK   8.572376  11.122933  5.898417   0.757348
6   3988.HK   0.087089   0.103901  2.040772   0.542531
7   0857.HK   0.452801   0.537192  6.220025   0.357093
8   0941.HK   2.807324   3.085197  3.532365  -3.961214
9   3968.HK   1.244907   1.440642  2.616969   0.477769
10  0883.HK   0.452254   0.664058  2.390609   0.781580
11  1088.HK   1.477430   1.850248  4.221930   0.723849
12  2318.HK   3.009922   3.494067  5.606126   0.364731
13  2899.HK   1.517178   1.856469  5.641739   0.881201
14  2628.HK   1.048789   1.315410  4.724412   0.827683
15  1299.HK   2.317408   3.031756  3.131564   0.537605
16  0300.HK   2.218